<center>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/SN_web_lightmode.png" width="300">
</center>


<h1>Analysis of Global COVID-19 Pandemic Data</h1>

Estimated time needed: **90** minutes



## Overview:

There are 10 tasks in this final project. All tasks will be graded by your peers who are also completing this assignment within the same session.

You need to submit the following the screenshot for the code and output for each task for review.

If you need to refresh your memories about specific coding details, you may refer to previous hands-on labs for code examples.


In [1]:
# This lab requires 'httr' and 'rvest'packages, which are already pre-loaded into this lab environment.
# However, if you are working on your local RStudio, please uncomment the below codes and install the packages.

#install.packages("httr")
#install.packages("rvest")

In [2]:
library(httr)
library(rvest)

Note: if you can import above libraries, please use install.packages() to install them first.


## TASK 1: Get a `COVID-19 pandemic` Wiki page using HTTP request


First, let's write a function to use HTTP request to get a public COVID-19 Wiki page.

Before you write the function, you can open this public page from this 

URL https://en.wikipedia.org/w/index.php?title=Template:COVID-19_testing_by_country using a web browser.

The goal of task 1 is to get the html page using HTTP request (`httr` library)


In [3]:
get_wiki_covid19_page <- function() {

    wiki_base_url <- "https://en.wikipedia.org/w/index.php"

    query_params <- list(
        title = "Template:COVID-19_testing_by_country"
    )

    response <- GET(
        url = wiki_base_url,
        query = query_params
    )

    return(response)
}


Call the `get_wiki_covid19_page` function to get a http response with the target html page


In [4]:
# Call the get_wiki_covid19_page function and print the response
response <- get_wiki_covid19_page()
print(response)

Response [https://en.wikipedia.org/w/index.php?title=Template%3ACOVID-19_testing_by_country]
  Date: 2026-08-03 19:11
  Status: 200
  Content-Type: text/html; charset=UTF-8
  Size: 973 kB
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-fea...
<head>
<meta charset="UTF-8">
<title>Template:COVID-19 testing by country - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-heade...
RLSTATE={"ext.globalCssJs.user.styles":"ready","site.styles":"ready","user.st...
<script>(RLQ=window.RLQ||[]).push(function(){mw.loader.impl(function(){return...
}];});});</script>
<link rel="stylesheet" href="/w/load.php?lang=en&amp;modules=ext.cite.parsoid...
...


## TASK 2: Extract COVID-19 testing data table from the wiki HTML page


On the COVID-19 testing wiki page, you should see a data table `<table>` node contains COVID-19 testing data by country on the page:

<a href="https://cognitiveclass.ai/">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/covid-19-by-country.png" width="400" align="center">
</a>

Note the numbers you actually see on your page may be different from above because it is still an on-going pandemic when creating this notebook.

The goal of task 2 is to extract above data table and convert it into a data frame


Now use the `read_html` function in rvest library to get the root html node from response


In [5]:
# Get the root html node from the http response in task 1 
root_html <- read_html(response)

Get the tables in the HTML root node using `html_nodes` function.


In [6]:
# Get the table node from the root html node
table_node <- html_nodes(root_html, "table")

Read the specific table from the multiple tables in the `table_node` using the `html_table` function and convert it into dataframe using `as.data.frame`

_Hint:- Please read the `table_node` with index 2(ex:- table_node[2])._


In [7]:
# Read the table node and convert it into a data frame, and print the data frame for review
covid_data_frame <- as.data.frame(
    html_table(table_node[2])
)

print(covid_data_frame)

## TASK 3: Pre-process and export the extracted data frame

The goal of task 3 is to pre-process the extracted data frame from the previous step, and export it as a csv file


Let's get a summary of the data frame


In [8]:
# Print the summary of the data frame
summary(covid_data_frame)

 Country.or.region    Date.a.             Tested            Units.b.        
 Length:173         Length:173         Length:173         Length:173        
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
 Confirmed.cases.   Confirmed..tested.. Tested..population..
 Length:173         Length:173          Length:173          
 Class :character   Class :character    Class :character    
 Mode  :character   Mode  :character    Mode  :character    
 Confirmed..population..     Ref.          
 Length:173              Length:173        
 Class :character        Class :character  
 Mode  :character        Mode  :character  

As you can see from the summary, the columns names are little bit different to understand and some column data types are not correct. For example, the `Tested` column shows as `character`. 

As such, the data frame read from HTML table will need some pre-processing such as removing irrelvant columns, renaming columns, and convert columns into proper data types.


We have prepared a pre-processing function for you to conver the data frame but you can also try to write one by yourself


In [9]:
preprocess_covid_data_frame <- function(data_frame) {
    
    shape <- dim(data_frame)

    # Remove the World row
    data_frame<-data_frame[!(data_frame$`Country.or.region`=="World"),]
    # Remove the last row
    data_frame <- data_frame[1:172, ]
    
    # We dont need the Units and Ref columns, so can be removed
    data_frame["Ref."] <- NULL
    data_frame["Units.b."] <- NULL
    
    # Renaming the columns
    names(data_frame) <- c("country", "date", "tested", "confirmed", "confirmed.tested.ratio", "tested.population.ratio", "confirmed.population.ratio")
    
    # Convert column data types
    data_frame$country <- as.factor(data_frame$country)
    data_frame$date <- as.factor(data_frame$date)
    data_frame$tested <- as.numeric(gsub(",","",data_frame$tested))
    data_frame$confirmed <- as.numeric(gsub(",","",data_frame$confirmed))
    data_frame$'confirmed.tested.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.tested.ratio`))
    data_frame$'tested.population.ratio' <- as.numeric(gsub(",","",data_frame$`tested.population.ratio`))
    data_frame$'confirmed.population.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.population.ratio`))
    
    return(data_frame)
}


Call the `preprocess_covid_data_frame` function


In [10]:
# call `preprocess_covid_data_frame` function and assign it to a new data frame
covid_data_frame <- preprocess_covid_data_frame(covid_data_frame)

Get the summary of the processed data frame again


In [11]:
# Print the summary of the processed data frame again
summary(covid_data_frame)

                country             date         tested         
 Afghanistan        :  1   2 Feb 2023 :  6   Min.   :     3880  
 Albania            :  1   1 Feb 2023 :  4   1st Qu.:   512037  
 Algeria            :  1   31 Jan 2023:  4   Median :  3029859  
 Andorra            :  1   1 Mar 2021 :  3   Mean   : 31377219  
 Angola             :  1   23 Jul 2021:  3   3rd Qu.: 12386725  
 Antigua and Barbuda:  1   29 Jan 2023:  3   Max.   :929349291  
 (Other)            :166   (Other)    :149                      
   confirmed        confirmed.tested.ratio tested.population.ratio
 Min.   :       0   Min.   : 0.00          Min.   :   0.0065      
 1st Qu.:   37839   1st Qu.: 5.00          1st Qu.:   9.4750      
 Median :  281196   Median :10.05          Median :  46.9500      
 Mean   : 2508340   Mean   :11.25          Mean   : 175.5043      
 3rd Qu.: 1278105   3rd Qu.:15.25          3rd Qu.: 156.5000      
 Max.   :90749469   Max.   :46.80          Max.   :3223.0000      
           

After pre-processing, you can see the columns and columns names are simplified, and columns types are converted into correct types.


The data frame has following columns:

- **country** - The name of the country
- **date** - Reported date
- **tested** - Total tested cases by the reported date
- **confirmed** - Total confirmed cases by the reported date
- **confirmed.tested.ratio** - The ratio of confirmed cases to the tested cases
- **tested.population.ratio** - The ratio of tested cases to the population of the country
- **confirmed.population.ratio** - The ratio of confirmed cases to the population of the country


OK, we can call `write.csv()` function to save the csv file into a file. 


In [12]:
# Export the data frame to a csv file
write.csv(
    covid_data_frame,
    "covid.csv",
    row.names = FALSE
)

Note for IBM Waston Studio, there is no traditional "hard disk" associated with a R workspace.

Even if you call `write.csv()` method to save the data frame as a csv file, it won't be shown in IBM Cloud Object Storage asset UI automatically.

However, you may still check if the `covid.csv` exists using following code snippet:


In [13]:
# Get working directory
wd <- getwd()
# Get exported 
file_path <- paste(wd, sep="", "/covid.csv")
# File path
print(file_path)
file.exists(file_path)

[1] "c:/Users/mel96/Documents/academic-lab/covid-2019-analysis/covid.csv"


[1] TRUE

**Optional Step**: If you have difficulties finishing above webscraping tasks, you may still continue with next tasks by downloading a provided csv file from here:


In [14]:
## Download a sample csv file
# covid_csv_file <- download.file("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/dataset/covid.csv", destfile="covid.csv")
# covid_data_frame_csv <- read.csv("covid.csv", header=TRUE, sep=",")

## TASK 4: Get a subset of the extracted data frame

The goal of task 4 is to get the 5th to 10th rows from the data frame with only `country` and `confirmed` columns selected


In [15]:
# Read covid_data_frame_csv from the csv file
covid_data_frame_csv <- read.csv(
  "covid.csv",
  header = TRUE,
  sep = ","
)
# Get the 5th to 10th rows, with two "country" "confirmed" columns
covid_data_frame_csv[5:10, c("country", "confirmed")]

,country,confirmed
,<chr>,<int>
5,Angola,20981
6,Antigua and Barbuda,832
7,Argentina,9060495
8,Armenia,422963
9,Australia,10112229
10,Austria,5789991


## TASK 5: Calculate worldwide COVID testing positive ratio

The goal of task 5 is to get the total confirmed and tested cases worldwide, and try to figure the overall positive ratio using `confirmed cases / tested cases`


In [16]:
# Get the total confirmed cases worldwide
total_confirmed <- sum(covid_data_frame$confirmed)
# Get the total tested cases worldwide
total_tested <- sum(covid_data_frame$tested)
# Get the positive ratio (confirmed / tested)
positive_ratio <- total_confirmed / total_tested
print(positive_ratio)

[1] 0.07994145


## TASK 6: Get a country list which reported their testing data 

The goal of task 6 is to get a catalog or sorted list of countries who have reported their COVID-19 testing data


In [17]:
# Get the `country` column
country <- covid_data_frame$country
# Check its class (should be Factor)
class(country)
# Conver the country column into character so that you can easily sort them
country <- as.character(country)
# Sort the countries AtoZ
country_AtoZ <- sort(country)
# Sort the countries ZtoA
country_ZtoA <- sort(country, decreasing = TRUE)
# Print the sorted ZtoA list
print(country_ZtoA)

[1] "factor"

  [1] "Zimbabwe"               "Zambia"                 "Vietnam"               
  [4] "Venezuela"              "Uzbekistan"             "Uruguay"               
  [7] "United States"          "United Kingdom"         "United Arab Emirates"  
 [10] "Ukraine"                "Uganda"                 "Turkey"                
 [13] "Tunisia"                "Trinidad and Tobago"    "Togo"                  
 [16] "Thailand"               "Tanzania"               "Taiwan[m]"             
 [19] "Switzerland[l]"         "Sweden"                 "Sudan"                 
 [22] "Sri Lanka"              "Spain"                  "South Sudan"           
 [25] "South Korea"            "South Africa"           "Slovenia"              
 [28] "Slovakia"               "Singapore"              "Serbia"                
 [31] "Senegal"                "Saudi Arabia"           "San Marino"            
 [34] "Saint Vincent"          "Saint Lucia"            "Saint Kitts and Nevis" 
 [37] "Rwanda"              

## TASK 7: Identify countries names with a specific pattern

The goal of task 7 is using a regular expression to find any countires start with `United`


In [18]:
# Use a regular expression `United.+` to find matches
country_pattern <- grep("United.+", country, value = TRUE)
# Print the matched country names
print(country_pattern)

[1] "United Arab Emirates" "United Kingdom"       "United States"       


## TASK 8: Pick two countries you are interested, and then review their testing data

The goal of task 8 is to compare the COVID-19 test data between two countires, you will need to select two rows from the dataframe, and select `country`, `confirmed`, `confirmed-population-ratio` columns


In [19]:
# Select a subset (should be only one row) of data frame based on a selected country name and columns
india <- covid_data_frame[
  covid_data_frame$country == "India",
  c("country", "confirmed", "confirmed.population.ratio")
]

# Select a subset (should be only one row) of data frame based on a selected country name and columns
usa <- covid_data_frame[
  covid_data_frame$country == "United States",
  c("country", "confirmed", "confirmed.population.ratio")
]

print(india)
print(usa)

   country confirmed confirmed.population.ratio
73   India  43585554                       31.7
          country confirmed confirmed.population.ratio
166 United States  90749469                       27.4


## TASK 9: Compare which one of the selected countries has a larger ratio of confirmed cases to population

The goal of task 9 is to find out which country you have selected before has larger ratio of confirmed cases to population, which may indicate that country has higher COVID-19 infection risk


In [22]:
# Use if-else statement
if (india$confirmed.population.ratio > usa$confirmed.population.ratio) {
#   print()
  print(as.character(india$country))
} else {
#   print()
  print(usa$country)

}


[1] "India"


## TASK 10: Find countries with confirmed to population ratio rate less than a threshold

The goal of task 10 is to find out which countries have the confirmed to population ratio less than 1%, it may indicate the risk of those countries are relatively low


In [ ]:
# Get a subset of any countries with `confirmed.population.ratio` less than the threshold
covid_data_frame[
  covid_data_frame$confirmed.population.ratio < 1,
]

                country        date    tested confirmed confirmed.tested.ratio
1           Afghanistan 17 Dec 2020    154767     49621                 32.100
3               Algeria  2 Nov 2020    230553     58574                 25.400
5                Angola  2 Feb 2021    399228     20981                  5.300
6   Antigua and Barbuda  6 Mar 2021     15268       832                  5.400
14           Bangladesh 24 Jul 2021   7417714   1151644                 15.500
19                Benin  4 May 2021    595112      7884                  1.300
25               Brunei  2 Aug 2021    153804       338                  0.220
27         Burkina Faso  4 Mar 2021    158777     12123                  7.600
28              Burundi  5 Jan 2021     90019       884                  0.980
29             Cambodia  1 Aug 2021   1812706     77914                  4.300
30             Cameroon 18 Feb 2021    942685     32681                  3.500
32                 Chad  2 Mar 2021     99027      4